# Notebook 6: Latency Proof — Customer Engineering Demo

This notebook is the customer-facing demo. Three sections:

| Section | What it shows |
|---|---|
| 6.1 Platform Setup | Account, region, service status |
| 6.2 Freshness Demo | Ingest a transaction → OFS feature updates in < 2s |
| 6.3 Latency Demo | 100 Thredd-format transactions scored via SPCS, p50/p95/p99 |

**Architecture**: All scoring happens inside SPCS. The OFS REST calls use
internal URLs (`*.svc.spcs.internal`) — the fastest possible path, same AWS VPC,
no public internet hop.

**Prerequisites:** nb01 through nb04 must be complete.


In [ ]:
import os, json, time, random, concurrent.futures
import numpy as np
import requests
from datetime import datetime, timezone
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql('USE ROLE FRAUD_MLOPS').collect()
session.sql('USE DATABASE FRAUD_DEMO_DEV').collect()
session.sql('USE SCHEMA FEATURES').collect()
session.sql('USE WAREHOUSE FRAUD_DEMO_WH').collect()

# Session token — works from Notebooks, no PAT env var needed
PAT = session.connection.rest.token or os.environ.get('SNOWFLAKE_PAT', '')
if not PAT:
    raise RuntimeError('Could not obtain session token.')

AUTH = {'Authorization': f'Snowflake Token="{PAT}"', 'Content-Type': 'application/json'}

# ── Verify platform ──────────────────────────────────────────────────────────
region  = session.sql('SELECT CURRENT_REGION()').collect()[0][0]
account = session.sql('SELECT CURRENT_ACCOUNT()').collect()[0][0]

print('=' * 60)
print('PLATFORM VERIFICATION')
print('=' * 60)
print(f'  Account : {account}')
print(f'  Region  : {region}')
print()
print('  Snowflake SPCS runs on AWS EC2 in this region.')
print('  The scoring container calls OFS on the internal SPCS mesh.')
print('  The backbone is the same AWS VPC as their architecture.')

# ── Scoring service endpoint ──────────────────────────────────────────────────
svc_rows = session.sql(
    "SHOW SERVICES LIKE 'FRAUD_SCORING_SERVICE' IN SCHEMA FRAUD_DEMO_PROD.ML"
).collect()
if not svc_rows:
    raise RuntimeError('FRAUD_SCORING_SERVICE not found. Run nb04_serving.ipynb first.')
SCORING_URL = f"https://{svc_rows[0]['dns_name']}"
svc_status  = svc_rows[0].get('status', '?')
print(f'\n  Scoring service : {svc_status}')
print(f'  Endpoint        : {SCORING_URL}')

# ── OFS public URL (for freshness demo from Notebooks) ────────────────────────
_raw = session.sql(
    "SELECT SYSTEM$GET_FEATURE_STORE_ONLINE_SERVICE_STATUS('FRAUD_DEMO_DEV.FEATURE_STORE')"
).collect()[0][0]
_svc = json.loads(_raw)
_eps = _svc.get('endpoints', {})

def _pick_url(ep_name):
    ep = _eps.get(ep_name, {})
    if isinstance(ep, str): return ep.rstrip('/')
    for key in ('public', 'Public', 'publicUrl', 'privatelink', 'private_link', 'internal'):
        if isinstance(ep, dict) and ep.get(key): return ep[key].rstrip('/')
    raise RuntimeError(f'No URL for {ep_name!r}')

OFS_QUERY_URL  = _pick_url('query')
OFS_INGEST_URL = _pick_url('ingest')
print(f'  OFS endpoint    : {OFS_QUERY_URL}  (public — for freshness demo)')
print()
print('Setup complete.')


## 6.2 Feature Freshness Demo

**The core selling point**: Snowflake's CONTINUOUS aggregation updates velocity features
within 2 seconds of ingest. The customer's Kinesis + batch pipeline takes minutes.

This means Snowflake can detect a card-testing attack (50 micro-transactions in 1 hour)
**in the 2-second window**. Their system can't — features are stale until the next batch.

This cell ingests one transaction and polls until the count feature increments.


In [ ]:
print('=' * 60)
print('FEATURE FRESHNESS DEMO')
print('=' * 60)

# Use brand-new entity keys to guarantee baseline is 0
test_cust  = f'DEMO_CUST_{random.randint(800000, 899999)}'
test_merch = f'DEMO_MERCH_{random.randint(800000, 899999)}'
test_dpan  = f'DEMO_DPAN_{random.randint(800000, 899999)}'
test_ip    = '185.23.44.100'
test_ts    = datetime.now(timezone.utc)

def ofs_query(fv_name, key_col, key_val, feature):
    r = requests.post(f'{OFS_QUERY_URL}/api/v1/query', headers=AUTH, json={
        'name': fv_name, 'version': 'V1', 'object_type': 'feature_view',
        'request_rows': [{'entity': {key_col: key_val}}],
        'features': [feature],
        'metadata_options': {'include_serving_status': True},
    }, timeout=10)
    if r.status_code not in (200, 207):
        raise RuntimeError(f'OFS query {fv_name}: {r.status_code} {r.text[:200]}')
    results = r.json().get('results', [])
    if not results: return None
    return results[0].get('features', [None])[0]

# Step 1: Verify baselines = 0
print('Baselines (expect null for unseen keys):')
FV_CHECK = [
    ('FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID', test_cust,  'PURCHASES_NUM_L1H'),
    ('FRAUD_MERCHANT_VELOCITY', 'MERCHANT_ID', test_merch, 'MERCHANT_TXN_NUM_L1H'),
    ('FRAUD_DPAN_VELOCITY',     'WALLET_DPAN', test_dpan,  'DPAN_TXN_NUM_L1H'),
    ('FRAUD_IP_VELOCITY',       'IP_ADDRESS',  test_ip,    'IP_TXN_NUM_L1H'),
]
for fv, key_col, key_val, feat in FV_CHECK:
    val = ofs_query(fv, key_col, key_val, feat)
    print(f'  {fv}: {feat} = {val}')

# Step 2: Ingest
print('\nIngesting transaction...')
ir = requests.post(f'{OFS_INGEST_URL}/api/v1/ingest', headers=AUTH, json={
    'dry_run': False,
    'records': {'FRAUD_TXN_EVENTS': [{
        'CUSTOMER_ID': test_cust, 'MERCHANT_ID': test_merch,
        'WALLET_DPAN': test_dpan, 'IP_ADDRESS':  test_ip,
        'AMOUNT_USD':  99.99, 'IS_GBR': 1.0,
        'EVENT_TS': test_ts.strftime('%Y-%m-%dT%H:%M:%SZ'),
    }]},
}, timeout=10)
if ir.status_code not in (200, 207):
    raise RuntimeError(f'Ingest failed: {ir.status_code}\n{ir.text[:500]}')
ingest_src = ir.json().get('sources', {}).get('FRAUD_TXN_EVENTS', {})
fv_results = ingest_src.get('feature_views', {}).get('results', [])
for r in fv_results:
    print(f'  FV {r["name"]}: {r["status"]}')
t_ingest = time.time()

# Step 3: Poll until all 4 FVs update
print('\nPolling (250ms intervals, 10s timeout)...')
latencies = {}
pending   = set(fv for fv, *_ in FV_CHECK)
for _ in range(40):
    time.sleep(0.25)
    for fv, key_col, key_val, feat in FV_CHECK:
        if fv in pending:
            val = ofs_query(fv, key_col, key_val, feat)
            if val is not None and val > 0:
                ms = (time.time() - t_ingest) * 1000
                latencies[fv] = ms
                pending.discard(fv)
                print(f'  {fv}: {ms:.0f}ms')
    if not pending:
        break

print('\n--- Results ---')
if pending:
    print(f'  TIMEOUT (>10s): {pending}')
if latencies:
    worst = max(latencies.values())
    print(f'  Slowest pipeline: {worst:.0f}ms')
    verdict = 'PASS' if worst < 2000 else 'FAIL'
    print(f'  Target < 2000ms : {verdict}')
    print()
    print('  CUSTOMER COMPARISON:')
    print(f'  Snowflake (CONTINUOUS OFS): {worst:.0f}ms freshness')
    print(f'  Their system  (Kinesis batch): minutes')
    print(f'  Result: Snowflake detects card-testing attacks in the same window.')
    print(f'          Their system cannot — velocity features are stale.')


## 6.3 End-to-End Scoring Latency

Sends 100 Thredd-format transactions to the SPCS scoring service via its public ingress.
Each request is fully scored: OFS internal feature lookup + XGBoost inference.

The **timing in each response comes from inside the SPCS container** — it shows the
internal OFS latency (not the Notebooks→SPCS HTTP overhead, which is excluded).


In [ ]:
print('=' * 60)
print('END-TO-END SCORING LATENCY (n=100)')
print('=' * 60)
print('Source: response body timing from SPCS container.')
print('OFS calls use internal URLs inside SPCS — fastest possible path.')
print()

# Sample 100 real entity keys from the transactions table
samples = session.sql('''
    SELECT CUSTOMER_ID, MERCHANT_ID, WALLET_DPAN, IP_ADDRESS,
           PURCHASE_AMOUNT, MERCHANT_COUNTRY, TRANSACTION_TS
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
    SAMPLE (110 ROWS)
''').collect()[:100]
print(f'Sampled {len(samples)} real transactions\n')

def build_thredd_payload(row):
    return {
        'Cust_Ref':      row['CUSTOMER_ID'],
        'Merchant_Id':   row['MERCHANT_ID'],
        'Token_Ref':     row['WALLET_DPAN'],
        'IP_Address':    row['IP_ADDRESS'],
        'Trans_Amount':  float(row['PURCHASE_AMOUNT'] or 0),
        'Merch_Country': str(row['MERCHANT_COUNTRY'] or 'GBR'),
        'Trans_DateTime': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    }

# Warmup: 10 requests to warm JIT and connection pool
print('Warming up (10 requests)...')
for row in samples[:10]:
    try:
        requests.post(f'{SCORING_URL}/score', headers=AUTH,
                      json=build_thredd_payload(row), timeout=15)
    except Exception:
        pass
print('  Warmup done.\n')

# Measured benchmark
print(f'Scoring {len(samples)} transactions...')
timings = []
for i, row in enumerate(samples):
    try:
        r = requests.post(f'{SCORING_URL}/score', headers=AUTH,
                          json=build_thredd_payload(row), timeout=15)
        r.raise_for_status()
        timings.append(r.json()['timing'])
    except Exception as exc:
        print(f'  Request {i} failed: {exc}')
    if (i + 1) % 25 == 0:
        print(f'  {i+1}/100 complete...')

# Results
def pct(arr, p): return round(float(np.percentile(arr, p)), 1)

ofs_arr   = [t['ofs_query_ms'] for t in timings]
xgb_arr   = [t['xgb_ms']       for t in timings]
total_arr = [t['total_ms']      for t in timings]

print()
print('─' * 62)
print(f'RESULTS  (n={len(timings)}, OFS internal URLs, inside SPCS)')
print('─' * 62)
print(f'  OFS feature lookup (5 FVs parallel)   p50: {pct(ofs_arr,50):5.1f}ms  p95: {pct(ofs_arr,95):5.1f}ms  p99: {pct(ofs_arr,99):5.1f}ms')
print(f'  XGBoost inference                     p50: {pct(xgb_arr,50):5.1f}ms  p95: {pct(xgb_arr,95):5.1f}ms')
print(f'  Total end-to-end                      p50: {pct(total_arr,50):5.1f}ms  p95: {pct(total_arr,95):5.1f}ms  p99: {pct(total_arr,99):5.1f}ms')

total_p50 = pct(total_arr, 50)
print()
print('─' * 62)
print('ARCHITECTURE COMPARISON')
print('─' * 62)
print(f'  Metric                   Customer (AWS)     Snowflake')
print(f'  Feature lookup           ~5-10ms DynamoDB   ~{pct(ofs_arr,50):.0f}ms OFS internal')
print(f'  Model inference          ~10-20ms Triton GPU ~{pct(xgb_arr,50):.0f}ms XGBoost CPU')
print(f'  Total p50                ~25-40ms            ~{total_p50:.0f}ms')
print(f'  Feature freshness        minutes (batch)     < 2s (CONTINUOUS)')
print(f'  EHI budget               < 50ms              {50-total_p50:.0f}ms headroom')
print(f'  Platform                 Kinesis+Lambda+      One Snowflake')
print(f'                           DynamoDB+SageMaker   account')
